# 00 — Data Exploration (EDA)

Analysis of both datasets before ML experiments.

In [ ]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # headless-safe backend
import matplotlib.pyplot as plt

# Handle both local (run from Notebooks/) and Colab execution
ROOT = Path("..").resolve()
if not (ROOT / "models").exists():
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/UrranQx/WaterMeterCV.git"], check=True)
    ROOT = Path("WaterMeterCV").resolve()

sys.path.insert(0, str(ROOT))

import pandas as pd
import cv2
import numpy as np
from models.data.unified_loader import load_water_meter_dataset, load_utility_meter_dataset
from models.utils.visualization import draw_roi_polygon, draw_digit_bboxes

print(f"ROOT: {ROOT}")

In [ ]:
wm_samples = load_water_meter_dataset(ROOT / "WaterMetricsDATA/waterMeterDataset/WaterMeters")
um_train = load_utility_meter_dataset(
    ROOT / "WaterMetricsDATA/utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11",
    split="train",
)
um_valid = load_utility_meter_dataset(
    ROOT / "WaterMetricsDATA/utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11",
    split="valid",
)
um_test = load_utility_meter_dataset(
    ROOT / "WaterMetricsDATA/utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11",
    split="test",
)

print(f"WaterMeter: {len(wm_samples)} samples")
print(f"UtilityMeter: {len(um_train)} train, {len(um_valid)} valid, {len(um_test)} test")

## Value Distribution (waterMeterDataset)

In [ ]:
wm_values = [s.value for s in wm_samples if s.value is not None]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(wm_values, bins=50)
axes[0].set_title("waterMeterDataset — value distribution")
axes[0].set_xlabel("Meter reading")

digit_counts = [len(str(v).replace(".", "").replace("-", "")) for v in wm_values]
axes[1].hist(digit_counts, bins=range(1, 12))
axes[1].set_title("Number of digits per reading")
plt.tight_layout()
plt.savefig(ROOT / "results" / "eda_value_distribution.png", dpi=150)
plt.close()
print(f"Value range: {min(wm_values):.3f} — {max(wm_values):.3f}")
print(f"Mean: {sum(wm_values)/len(wm_values):.3f}")

## ROI Polygon Samples (waterMeterDataset)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, sample in zip(axes.flat, wm_samples[:8]):
    img = cv2.imread(str(sample.image_path))
    if img is None:
        ax.set_title("image not found")
        ax.axis("off")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if sample.roi_polygon:
        img = draw_roi_polygon(img, sample.roi_polygon)
    ax.imshow(img)
    ax.set_title(f"val={sample.value}")
    ax.axis("off")
plt.suptitle("waterMeterDataset — ROI polygons")
plt.tight_layout()
plt.savefig(ROOT / "results" / "eda_roi_samples.png", dpi=150)
plt.close()
print("ROI samples saved")

## Digit BBox Samples (utility-meter)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, sample in zip(axes.flat, um_train[:8]):
    img = cv2.imread(str(sample.image_path))
    if img is None:
        ax.set_title("image not found")
        ax.axis("off")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if sample.digit_bboxes:
        img = draw_digit_bboxes(img, sample.digit_bboxes)
    ax.imshow(img)
    ax.set_title(f"val={sample.value}")
    ax.axis("off")
plt.suptitle("utility-meter — digit bounding boxes")
plt.tight_layout()
plt.savefig(ROOT / "results" / "eda_digit_bboxes.png", dpi=150)
plt.close()
print("Digit bbox samples saved")

## Digit Class Distribution (utility-meter train)

In [ ]:
from collections import Counter

all_digits = []
for s in um_train:
    if s.digit_bboxes:
        all_digits.extend([d[0] for d in s.digit_bboxes])

counts = Counter(all_digits)
plt.figure(figsize=(10, 5))
plt.bar([str(k) for k in sorted(counts.keys())], [counts[k] for k in sorted(counts.keys())])
plt.title("Digit class distribution (utility-meter train)")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.savefig(ROOT / "results" / "eda_digit_distribution.png", dpi=150)
plt.close()
print(f"Total digit annotations: {len(all_digits)}")
print("Distribution:", dict(sorted(counts.items())))

## ROI Size Analysis (waterMeterDataset)

In [ ]:
from shapely.geometry import Polygon

areas = []
for s in wm_samples:
    if s.roi_polygon and len(s.roi_polygon) >= 3:
        areas.append(Polygon(s.roi_polygon).area)

plt.figure(figsize=(10, 5))
plt.hist(areas, bins=30)
plt.title("ROI polygon area distribution (normalized coords)")
plt.xlabel("Area")
plt.ylabel("Count")
plt.savefig(ROOT / "results" / "eda_roi_areas.png", dpi=150)
plt.close()
print(f"Median ROI area: {sorted(areas)[len(areas)//2]:.4f}")
print(f"Area range: {min(areas):.4f} — {max(areas):.4f}")

## Выводы EDA

*(Заполнить после запуска)*

- **Размер датасетов:** waterMeter=?, utility-meter=?
- **Распределение значений:** ...
- **Распределение классов цифр:** ...
- **Типичный размер ROI:** ...
- **Потенциальные проблемы:** ...